# V3 Category Handler Test Runner

Exercise the new Step 3 → category-handler endpoint path for representative categories. Each endpoint cell builds a sample `CategoryCall` plus its parent Step-2 `Trait`, runs `search_v2.endpoint_fetching.category_handlers.run_handler`, and prints the fired endpoint wrappers, handler buckets, top scored candidates, and elapsed time.

The focused cells below cover the endpoints currently migrated through the category-handler system: entity, franchise, metadata, award, and studio. Older direct generator cells are intentionally not used for those endpoints.


## Setup

Run this cell first. It imports shared category-handler inputs, opens Postgres / Redis / Qdrant, and leaves endpoint-specific imports inside each cell only when they are still direct-executor examples.


In [3]:
import sys, time, asyncio
from datetime import date
from pathlib import Path

# Ensure project root is on the path so imports resolve regardless of where the
# notebook is launched from.
project_root = str(Path.cwd())
if project_root not in sys.path:
    sys.path.insert(0, project_root)

from dotenv import load_dotenv
load_dotenv(Path(project_root) / ".env")

from implementation.llms.generic_methods import LLMProvider
from schemas.enums import Polarity, Role
from schemas.step_2 import Trait
from schemas.step_3 import CategoryCall
from schemas.trait_category import CategoryName
from search_v2.endpoint_fetching.category_handlers.handler import run_handler

# Direct endpoint generators / executors are imported inside the cells that still
# use them. The migrated endpoint cells below exercise the category-handler path.

# Database connections — mirrors the FastAPI lifespan in api/main.py.
from db.postgres import pool as postgres_pool
from db.redis import init_redis, check_redis
from db.qdrant import check_qdrant, qdrant_client
import db.redis as _redis_module

# Idempotent: re-running the setup cell must not double-open the Postgres pool
# or leak a prior Redis client.
if postgres_pool._closed:
    await postgres_pool.open()
    print("Postgres: pool opened")
else:
    print("Postgres: pool already open")

if _redis_module._redis_client is None:
    await init_redis()
    print(f"Redis:    initialized ({await check_redis()})")
else:
    print(f"Redis:    already initialized ({await check_redis()})")

print(f"Qdrant:   {await check_qdrant()}")


Postgres: pool already open
Redis:    already initialized (ok)
Qdrant:   ok


## Configuration

Shared date and optional legacy LLM config. The category-handler path is internally configured to use `gpt-5.4-mini` with no reasoning; the direct semantic / keyword cells still read the provider/model variables.


In [4]:
# ---- Presets (uncomment one) ----

# Kimi K2.5 — no thinking
# provider, model, kwargs = LLMProvider.KIMI, "kimi-k2.5", {"enable_thinking": False}

# GPT 5.4 Mini — no thinking
provider, model, kwargs = LLMProvider.OPENAI, "gpt-5.4-mini", {"reasoning_effort": "none", "verbosity": "low"}

# Gemini 3 Flash — no thinking
# provider, model, kwargs = LLMProvider.GEMINI, "gemini-3-flash-preview", {"thinking_config": {"thinking_budget": 0}}

# Today's date — useful for direct metadata / award generator experiments.
today = date.today()

print(f"Provider: {provider.value}")
print(f"Model:    {model}")
print(f"Kwargs:   {kwargs or '(defaults)'}")
print(f"Today:    {today.isoformat()}")


Provider: openai
Model:    gpt-5.4-mini
Kwargs:   {'reasoning_effort': 'none', 'verbosity': 'low'}
Today:    2026-04-30


## Result Formatting Helpers

`print_top_results` bulk-fetches movie cards for scored candidates. `print_handler_result` shows the category-handler output buckets plus any fired endpoint wrappers from the new system.


In [6]:
from datetime import datetime, timezone
from typing import Iterable, Protocol

from db.postgres import fetch_movie_cards

TOP_N = 15


class _Scored(Protocol):
    movie_id: int
    score: float


async def print_top_results(
    candidates: Iterable[_Scored],
    *,
    limit: int = TOP_N,
    sort_desc: bool = True,
) -> None:
    """Print scored movies as ``score  title (year)``, one per line.

    Bulk-fetches card metadata in a single query (per the
    "never query Postgres per-candidate" invariant in AGENTS.md).
    """
    items = list(candidates)
    if not items:
        print("(no scored candidates)")
        return

    if sort_desc:
        items = sorted(items, key=lambda c: c.score, reverse=True)
    items = items[:limit]

    cards = await fetch_movie_cards([c.movie_id for c in items])
    card_by_id = {c["movie_id"]: c for c in cards}

    for cand in items:
        card = card_by_id.get(cand.movie_id)
        if card is None:
            title, year = "<missing card>", "?"
        else:
            title = card["title"] or "<untitled>"
            ts = card["release_ts"]
            year = (
                datetime.fromtimestamp(ts, tz=timezone.utc).year
                if ts is not None else "?"
            )
        print(f"  {cand.score:.3f}  {title} ({year})")


def make_trait(
    *,
    surface_text: str,
    evaluative_intent: str,
    contextualized_phrase: str | None = None,
    role: Role = Role.CARVER,
    polarity: Polarity = Polarity.POSITIVE,
    salience: str = "central",
) -> Trait:
    """Small notebook factory for parent Step-2 Trait context.

    The handler stamps role/polarity from this object onto every fired endpoint
    wrapper. The remaining fields are included because they are required by the
    Step-2 schema, even though run_handler only reads role and polarity today.
    """
    return Trait(
        surface_text=surface_text,
        evaluative_intent=evaluative_intent,
        qualifier_relation="n/a",
        anchor_reference="n/a",
        role_evidence="The sample trait is framed as an eligibility criterion for this notebook probe.",
        role=role,
        polarity=polarity,
        relevance_to_query="This notebook sample treats the trait as the central endpoint behavior under inspection.",
        salience=salience,
        contextualized_phrase=contextualized_phrase or surface_text,
    )


def make_category_call(
    *,
    category: CategoryName,
    expressions: list[str],
    retrieval_intent: str,
) -> CategoryCall:
    return CategoryCall(
        category=category,
        expressions=expressions,
        retrieval_intent=retrieval_intent,
    )


def _print_score_map(title: str, score_map: dict[int, float]) -> None:
    print(f"{title}: {len(score_map)}")
    if score_map:
        top = sorted(score_map.items(), key=lambda item: item[1], reverse=True)[:10]
        for movie_id, score in top:
            print(f"  {score:.3f}  {movie_id}")


async def print_handler_result(handler_result, *, elapsed: float) -> None:
    print(f"Handler completed in {elapsed:.2f}s")
    print(f"Category: {handler_result.category.name if handler_result.category else '(none)'}")
    print()

    print("=" * 60)
    print("FIRED ENDPOINT WRAPPERS")
    print("=" * 60)
    if not handler_result.fired_endpoints:
        print("(none)")
    for route, wrapper in handler_result.fired_endpoints:
        print(f"\n--- {route.value} / {type(wrapper).__name__} ---")
        print(wrapper.model_dump_json(indent=2))

    print("\n" + "=" * 60)
    print("HANDLER BUCKETS")
    print("=" * 60)
    _print_score_map("inclusion_candidates", handler_result.inclusion_candidates)
    _print_score_map("downrank_candidates", handler_result.downrank_candidates)
    print(f"exclusion_ids: {len(handler_result.exclusion_ids)}")
    print(f"preference_specs: {len(handler_result.preference_specs)}")

    print("\n" + "=" * 60)
    print("TOP INCLUSION CANDIDATES")
    print("=" * 60)
    pseudo_scores = [type("Scored", (), {"movie_id": mid, "score": score})() for mid, score in handler_result.inclusion_candidates.items()]
    await print_top_results(pseudo_scores)


## Endpoint 1: Entity via Category Handler

Runs a sample `NAMED_CHARACTER` category call through the new handler system. The handler LLM emits an `EntityEndpointParameters` wrapper whose `.parameters` payload is one of the entity executor specs, then the handler executes it as a candidate-generating positive carver.


In [ ]:
entity_trait = make_trait(
    surface_text="The Rock",
    evaluative_intent="Identify and retrieve movies featuring the actor Dwayne 'The Rock' Johnson as a primary cast member.",
)
entity_call = make_category_call(
    category=CategoryName.PERSON_CREDIT,
    expressions=["Dwayne Johnson"],
    retrieval_intent=(
        "Retrieve the filmography of the actor Dwayne Johnson. The search should look for an exact match within the indexed actor credits to ensure the results are movies where he is a primary cast member."
    ),
)

start = time.perf_counter()
entity_handler_result = await run_handler(
    category_call=entity_call,
    trait=entity_trait,
    qdrant_client=qdrant_client,
)
entity_elapsed = time.perf_counter() - start
await print_handler_result(entity_handler_result, elapsed=entity_elapsed)


FileNotFoundError: Missing additional_objective_notes chunk for CategoryName.PERSON_CREDIT. Expected at: /Users/michaelkeohane/Documents/movie-finder-rag/search_v2/endpoint_fetching/category_handlers/prompts/categories/additional_objective_notes/person_credit.md

## Endpoint 2: Franchise via Category Handler

Runs a sample `FRANCHISE_LINEAGE` category call through the new handler system. The LLM emits a `FranchiseEndpointParameters` wrapper, and the handler executes the franchise endpoint.


In [ ]:
franchise_trait = make_trait(
    surface_text="Shrek movies",
    evaluative_intent="The user wants movies belonging to the Shrek franchise or shared universe, with main-line lineage preferred where applicable.",
)
franchise_call = make_category_call(
    category=CategoryName.FRANCHISE_LINEAGE,
    expressions=["Shrek franchise", "main-line Shrek lineage"],
    retrieval_intent=(
        "Search for movies in the Shrek franchise / shared universe. Treat the "
        "franchise name as the primary lookup target and preserve the main-line "
        "lineage preference as a scoring bias rather than excluding related entries."
    ),
)

start = time.perf_counter()
franchise_handler_result = await run_handler(
    category_call=franchise_call,
    trait=franchise_trait,
    qdrant_client=qdrant_client,
)
franchise_elapsed = time.perf_counter() - start
await print_handler_result(franchise_handler_result, elapsed=franchise_elapsed)


## Endpoint 3: Keyword

Translates a classification / tag requirement into a `KeywordQuerySpec` (one `UnifiedClassification` registry pick) and executes the backing posting-list lookup.

In [ ]:
from search_v2.endpoint_fetching.keyword_query_generation import generate_keyword_query
from search_v2.endpoint_fetching.keyword_query_execution import execute_keyword_query

keyword_inputs = {
    "intent_rewrite": "Find animated movies for kids",
    "description": "is an animated movie",
    "route_rationale": "genre classification",
}

keyword_candidate_ids: set[int] = set()

# ---- Generation ----
start = time.perf_counter()
keyword_spec, keyword_in_tok, keyword_out_tok = await generate_keyword_query(
    intent_rewrite=keyword_inputs["intent_rewrite"],
    description=keyword_inputs["description"],
    route_rationale=keyword_inputs["route_rationale"],
    provider=provider,
    model=model,
    **kwargs,
)
keyword_gen_elapsed = time.perf_counter() - start

print(f"Generation completed in {keyword_gen_elapsed:.2f}s")
print(f"Tokens — input: {keyword_in_tok:,}  output: {keyword_out_tok:,}\n")
print("=" * 60)
print(f"LLM RESULT — {type(keyword_spec).__name__}")
print("=" * 60)
print(keyword_spec.model_dump_json(indent=2))

# ---- Execution ----
restrict = keyword_candidate_ids or None
start = time.perf_counter()
keyword_result = await execute_keyword_query(keyword_spec, restrict_to_movie_ids=restrict)
keyword_exec_elapsed = time.perf_counter() - start

print()
print("=" * 60)
print(f"EXECUTION RESULT [{'pool' if restrict else 'unrestricted'}] — {len(keyword_result.scores)} movies in {keyword_exec_elapsed:.2f}s")
print("=" * 60)
await print_top_results(keyword_result.scores)

## Endpoint 4: Metadata via Category Handler

Runs a sample `NUMERIC_RECEPTION_SCORE` category call through the new handler system. The LLM emits a `MetadataEndpointParameters` wrapper whose payload can populate one or more structured metadata columns.


In [ ]:
metadata_trait = make_trait(
    surface_text="low-rated or poorly reviewed",
    evaluative_intent="The user wants movies with weak numeric reception signals, such as low critic or audience ratings.",
)
metadata_call = make_category_call(
    category=CategoryName.NUMERIC_RECEPTION_SCORE,
    expressions=["low audience rating", "poor critic reception"],
    retrieval_intent=(
        "Search for movies with low or poor numeric reception. The expressions "
        "are substitutable reception signals, so a movie can satisfy the call via "
        "low audience rating, poor critic reception, or both."
    ),
)

start = time.perf_counter()
metadata_handler_result = await run_handler(
    category_call=metadata_call,
    trait=metadata_trait,
    qdrant_client=qdrant_client,
)
metadata_elapsed = time.perf_counter() - start
await print_handler_result(metadata_handler_result, elapsed=metadata_elapsed)


## Endpoint 5: Award via Category Handler

Runs a sample `AWARDS` category call through the new handler system. The LLM emits an `AwardEndpointParameters` wrapper, and the handler executes the award endpoint.


In [ ]:
award_trait = make_trait(
    surface_text="won at least one Oscar",
    evaluative_intent="The user wants movies with at least one Academy Award win.",
)
award_call = make_category_call(
    category=CategoryName.AWARDS,
    expressions=["Oscar winner", "at least one win"],
    retrieval_intent=(
        "Search for movies that won at least one Oscar / Academy Award. This is "
        "one award condition with a winner outcome and a minimum count of one."
    ),
)

start = time.perf_counter()
award_handler_result = await run_handler(
    category_call=award_call,
    trait=award_trait,
    qdrant_client=qdrant_client,
)
award_elapsed = time.perf_counter() - start
await print_handler_result(award_handler_result, elapsed=award_elapsed)


## Endpoint 6: Studio via Category Handler

Runs a sample `STUDIO_BRAND` category call through the new handler system. The LLM emits a `StudioEndpointParameters` wrapper, and the handler executes the studio endpoint.


In [ ]:
studio_trait = make_trait(
    surface_text="Disney films",
    evaluative_intent="The user wants movies produced by the Disney studio brand / production company family.",
)
studio_call = make_category_call(
    category=CategoryName.STUDIO_BRAND,
    expressions=["Disney"],
    retrieval_intent=(
        "Search for films produced by Disney as a production-company / studio "
        "brand. Treat Disney as an umbrella studio-brand lookup, not streaming "
        "availability on Disney+."
    ),
)

start = time.perf_counter()
studio_handler_result = await run_handler(
    category_call=studio_call,
    trait=studio_trait,
    qdrant_client=qdrant_client,
)
studio_elapsed = time.perf_counter() - start
await print_handler_result(studio_handler_result, elapsed=studio_elapsed)


## Endpoint 7: Media Type

Media-type has no LLM translator — `MediaTypeQuerySpec` is built deterministically (regex match against Step-3 expressions in production; constructed by hand here). Edit `media_type_formats` to choose which `ReleaseFormat` members to filter on.

In [15]:
from search_v2.endpoint_fetching.media_type_query_execution import execute_media_type_query
from schemas.enums import ReleaseFormat
from schemas.media_type_translation import MediaTypeQuerySpec
from search_v2.endpoint_fetching.category_handlers.media_type_router import (
    build_media_type_query_spec,
)

media_type_candidate_ids: set[int] = set()

media_type_call = make_category_call(
    category=CategoryName.MEDIA_TYPE,
    expressions=[
        "TV movies",
        "shorts"
      ],
    retrieval_intent=(
        "Identify films whose primary distribution format is television. This call gates the population to include only those movies produced for broadcast, cable, or streaming-first television networks, distinguishing them from theatrical feature films."
    ),
)

media_type_spec = build_media_type_query_spec(
    category_call=media_type_call
)

print("=" * 60)
print(f"SPEC — {type(media_type_spec).__name__} (deterministic, no LLM)")
print("=" * 60)
print(media_type_spec.model_dump_json(indent=2))

# ---- Execution ----
restrict = media_type_candidate_ids or None
start = time.perf_counter()
media_type_result = await execute_media_type_query(media_type_spec, restrict_to_movie_ids=restrict)
media_type_exec_elapsed = time.perf_counter() - start

print()
print("=" * 60)
print(f"EXECUTION RESULT [{'pool' if restrict else 'unrestricted'}] — {len(media_type_result.scores)} movies in {media_type_exec_elapsed:.2f}s")
print("=" * 60)
await print_top_results(media_type_result.scores)

SPEC — MediaTypeQuerySpec (deterministic, no LLM)
{
  "thinking": "Deterministic MEDIA_TYPE phrase match resolved TV_MOVIE, SHORT from the category-call expressions.",
  "formats": [
    "tvMovie",
    "short"
  ]
}

EXECUTION RESULT [unrestricted] — 14913 movies in 0.18s
  1.000  Period. End of Sentence. (2018)
  1.000  Nobody Will Believe You (2021)
  1.000  Romance of a Jewess (1908)
  1.000  Love on Repeat (2019)
  1.000  Dial M for Middlesbrough (2019)
  1.000  Blitz Wolf (1942)
  1.000  The Curtain Pole (1909)
  1.000  Come Dance at My Wedding (2009)
  1.000  The Good Witch (2008)
  1.000  Pinocchio (1976)
  1.000  The Angel of Pennsylvania Avenue (1996)
  1.000  Father Gets in the Game (1908)
  1.000  Making Waves (2023)
  1.000  Never Too Late to Celebrate (2023)
  1.000  Napa Ever After (2023)


## Endpoint 8: Trending

Trending has no LLM translator — execution reads the precomputed trending hash from Redis directly. The only "input" is an optional candidate-pool restriction.

In [7]:
from search_v2.endpoint_fetching.trending_query_execution import execute_trending_query

trending_candidate_ids: set[int] = set()

# ---- Execution only (no generation step) ----
restrict = trending_candidate_ids or None
start = time.perf_counter()
trending_result = await execute_trending_query(restrict_to_movie_ids=restrict)
trending_exec_elapsed = time.perf_counter() - start

print("=" * 60)
print("SPEC — (none; trending has no LLM translator and no spec object)")
print("=" * 60)
print()
print("=" * 60)
print(f"EXECUTION RESULT [{'pool' if restrict else 'unrestricted'}] — {len(trending_result.scores)} movies in {trending_exec_elapsed:.2f}s")
print("=" * 60)
await print_top_results(trending_result.scores)

SPEC — (none; trending has no LLM translator and no spec object)

EXECUTION RESULT [unrestricted] — 489 movies in 0.02s
  0.978  <missing card> (?)
  0.968  <missing card> (?)
  0.961  <missing card> (?)
  0.955  <missing card> (?)
  0.950  <missing card> (?)
  0.945  <missing card> (?)
  0.941  <missing card> (?)
  0.937  Greenland 2: Migration (2026)
  0.933  Avatar: Fire and Ash (2025)
  0.929  <missing card> (?)
  0.926  <missing card> (?)
  0.922  <missing card> (?)
  0.919  <missing card> (?)
  0.916  Venganza (2026)
  0.913  <missing card> (?)


## Endpoint 9a: Semantic — Dealbreaker (CARVER role)

Translates a single positive-presence semantic trait into a `SemanticParameters` payload — one LLM call inventories qualifiers, populates one or more `space_queries` entries (one per applicable vector space), and commits a `primary_vector`. Execution uses `Role.CARVER`, which consults only the entry whose space matches `primary_vector`.

In [ ]:
from search_v2.endpoint_fetching.semantic_query_generation import generate_semantic_dealbreaker_query
from search_v2.endpoint_fetching.semantic_query_execution import execute_semantic_query
from schemas.enums import Role

semantic_dealbreaker_inputs = {
    "intent_rewrite": "Find slow-burn psychological thrillers with an unreliable narrator",
    "description": "features an unreliable narrator as a central storytelling device",
    "route_rationale": "narrative craft / storytelling technique",
}

# Empty set → D2 (probe generates its own pool); populate → D1 (pool-scoring).
semantic_dealbreaker_candidate_ids: set[int] = set()

# ---- Generation ----
start = time.perf_counter()
semantic_db_spec, semantic_db_in_tok, semantic_db_out_tok = await generate_semantic_dealbreaker_query(
    intent_rewrite=semantic_dealbreaker_inputs["intent_rewrite"],
    description=semantic_dealbreaker_inputs["description"],
    route_rationale=semantic_dealbreaker_inputs["route_rationale"],
    provider=provider,
    model=model,
    **kwargs,
)
semantic_db_gen_elapsed = time.perf_counter() - start

print(f"Generation completed in {semantic_db_gen_elapsed:.2f}s")
print(f"Tokens — input: {semantic_db_in_tok:,}  output: {semantic_db_out_tok:,}\n")
print("=" * 60)
print(f"LLM RESULT — {type(semantic_db_spec).__name__}")
print("=" * 60)
print(semantic_db_spec.model_dump_json(indent=2))

# Generators may emit either `SemanticParameters` directly or a wrapper with a
# `.parameters` attribute. Accept both shapes so the cell stays robust.
semantic_db_params = getattr(semantic_db_spec, "parameters", semantic_db_spec)

# ---- Execution (CARVER role uses primary_vector only) ----
restrict = semantic_dealbreaker_candidate_ids or None
start = time.perf_counter()
semantic_db_result = await execute_semantic_query(
    semantic_db_params,
    role=Role.CARVER,
    restrict_to_movie_ids=restrict,
    qdrant_client=qdrant_client,
)
semantic_db_exec_elapsed = time.perf_counter() - start

scenario = "D1 (pre-built pool)" if restrict else "D2 (probe generates candidates)"
print()
print("=" * 60)
print(f"EXECUTION RESULT [{scenario}] — {len(semantic_db_result.scores)} movies in {semantic_db_exec_elapsed:.2f}s")
print("=" * 60)
await print_top_results(semantic_db_result.scores)

## Endpoint 9b: Semantic — Preference (QUALIFIER role)

Translates a grouped preference description into a `SemanticParameters` payload — decomposes into atomic qualifiers, picks 1+ of the 7 vector spaces, assigns each a `SpaceWeight` (`central` / `supporting`), and emits one body per selected space. Execution uses `Role.QUALIFIER`, which consults every populated space weighted accordingly (`primary_vector` is ignored).

In [ ]:
from search_v2.endpoint_fetching.semantic_query_generation import generate_semantic_preference_query
from search_v2.endpoint_fetching.semantic_query_execution import execute_semantic_query
from schemas.enums import Role

semantic_preference_inputs = {
    "intent_rewrite": "Something slow-burn, atmospheric, and melancholy for a rainy Sunday afternoon",
    "description": "slow-burn, atmospheric, rainy-day melancholy for a quiet Sunday afternoon",
    "route_rationale": "viewing mood / tone preference",
}

# Empty set → P2 (probes generate pool); populate → P1 (pool-scoring).
semantic_preference_candidate_ids: set[int] = set()

# ---- Generation ----
start = time.perf_counter()
semantic_pref_spec, semantic_pref_in_tok, semantic_pref_out_tok = await generate_semantic_preference_query(
    intent_rewrite=semantic_preference_inputs["intent_rewrite"],
    description=semantic_preference_inputs["description"],
    route_rationale=semantic_preference_inputs["route_rationale"],
    provider=provider,
    model=model,
    **kwargs,
)
semantic_pref_gen_elapsed = time.perf_counter() - start

print(f"Generation completed in {semantic_pref_gen_elapsed:.2f}s")
print(f"Tokens — input: {semantic_pref_in_tok:,}  output: {semantic_pref_out_tok:,}\n")
print("=" * 60)
print(f"LLM RESULT — {type(semantic_pref_spec).__name__}")
print("=" * 60)
print(semantic_pref_spec.model_dump_json(indent=2))

# Tolerate both shapes — see the dealbreaker cell for rationale.
semantic_pref_params = getattr(semantic_pref_spec, "parameters", semantic_pref_spec)

# ---- Execution (QUALIFIER role uses every populated space, weighted) ----
restrict = semantic_preference_candidate_ids or None
start = time.perf_counter()
semantic_pref_result = await execute_semantic_query(
    semantic_pref_params,
    role=Role.QUALIFIER,
    restrict_to_movie_ids=restrict,
    qdrant_client=qdrant_client,
)
semantic_pref_exec_elapsed = time.perf_counter() - start

scenario = "P1 (pre-built pool)" if restrict else "P2 (probes generate candidates)"
print()
print("=" * 60)
print(f"EXECUTION RESULT [{scenario}] — {len(semantic_pref_result.scores)} movies in {semantic_pref_exec_elapsed:.2f}s")
print("=" * 60)
await print_top_results(semantic_pref_result.scores)